# Python UDFs — seeing inside the black box

A user-defined function is opaque to plan analysis. `WHERE classify(amount) = 'high'`
is one call, over one column, producing one value: no branches to score, no constraints
to solve, and every argument equally implicated in the result.

For a **Python** UDF the code is Python source, and it can be read. This notebook plants
one fault and shows what three techniques can say about it once the function is no
longer a black box.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data and the functions

Twelve orders across three customers. One of them, `o8`, is an outlier at `99999` —
every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, StringType
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("python-udf-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
orders.createOrReplaceTempView("orders")


def classify(amount):
    if amount > 1000:
        return "high"
    elif amount > 100:
        return "medium"
    return "low"


def band(amount):
    if amount > 1000:
        return -amount          # the planted fault: an outlier flips sign
    return amount


def score(amount, note):
    return amount * 2           # `note` is passed and never read


spark.udf.register("classify", classify, StringType())
spark.udf.register("band", band, IntegerType())
spark.udf.register("score", score, IntegerType())

orders.show()

## Reading the function

`register` parses the function's own source and records what it found. Nothing else
about how you write queries changes.

In [ ]:
profile = bigasterisk.udf.register(spark, classify)

print(profile)
print("\nbranches:")
for condition, params in profile.branches:
    print("  ", condition, " reads", sorted(params))
print("\npaths:")
for constraint, returns, exact in profile.paths:
    print("  ", constraint, "->", returns, "" if exact else "(approximate)")
print("\ninfluencing:", sorted(profile.influencing))

## Test generation solves through it

A condition on a UDF's *result* cannot be solved — no solver inverts an opaque call.
With a profile, the conditions under which the function returns `'high'` are ordinary
comparisons on `amount`, and those can be solved. Every generated input is executed and
the path it was built for is checked.

In [ ]:
QUERY = "SELECT oid FROM orders WHERE classify(amount) = 'high'"
testgen = bigasterisk.testgen(spark)

suite = testgen.generate(QUERY, {"orders": orders}, seed=1)
for case in suite.cases:
    print(case)

# the same query with the function left as a black box
bigasterisk.udf.unregister(spark, "classify")
blind = testgen.generate(QUERY, {"orders": orders}, seed=1)
print("\nwithout a profile:")
for case in blind.cases:
    print("  ", case.path, "->", case.note)

bigasterisk.udf.register(spark, classify)

## Operation isolation ranks a branch inside it

`band` negates amounts over 1000, so the outlier comes out negative. The branch that
does it is inside the function, invisible to the plan — until it is bound to the column
the call site passes.

In [ ]:
bigasterisk.udf.register(spark, band)
faulty = spark.sql("SELECT oid, band(amount) AS value FROM orders")
faulty.show(3)

result = bigasterisk.optdebug(spark).localize(faulty, "value < 0")
for operation in result.ranked[:5]:
    print(operation)

## Influence names the columns that mattered

`score` takes two arguments and reads one. Which record influenced the result is half
the answer; which of its columns did is the other half.

In [ ]:
bigasterisk.udf.register(spark, score)
ranked = bigasterisk.influence(spark).influencers(
    "SELECT cid, MAX(score(amount, oid)) AS peak FROM orders GROUP BY cid",
    faulty_where="peak > 1000")

print(ranked[0])
print("columns that could reach the result:", sorted(ranked[0].columns))

## What it refuses

Anything the analysis cannot read is reported rather than guessed at. An inexact path is
never solved through, because a subtly wrong constraint would generate a test that proves
nothing.

In [ ]:
def unreadable(text):
    if text.encode("utf8") == b"x":
        return "odd"
    return "even"


partial = bigasterisk.udf.analyze(unreadable)
print(partial)
print("could not read:", partial.unsupported)
print("solvable:", partial.solvable)

## Check

In [ ]:
assert profile.solvable
assert sorted(profile.influencing) == ["amount"]
assert len(profile.paths) == 3

assert any(case.verified for case in suite.cases), [c.note for c in suite.cases]
assert all(not case.verified for case in blind.cases)

assert result.ranked[0].branch is not None and "1000" in result.ranked[0].branch
assert ranked[0].columns == {"amount"}
assert ranked[0].narrowed

assert not partial.complete and not partial.solvable
assert len(partial.unsupported) == 1
print("OK")